# A neuron with calcium dynamics and spike-frequency adaptation due to a Ca2+-activated K+ channel

**Calcium is good for you!**
![image](https://media.istockphoto.com/photos/glass-of-milk-picture-id1206080627?k=20&m=1206080627&s=612x612&w=0&h=NfdmNI8WYa5Kd7zMCqpZ8hFkakQCWzkv9aD9r5yhdRw=)

Everything so far has been squid: sodium in, potassium out, repeat. Real
neurons also keep track of **how much they have already fired**, and calcium
is how they do it. Every action potential lets a little Ca2+ in; it takes
time for the cell to clear it; and while it is there it opens SK channels that slow the cell down.

That is **spike-frequency adaptation**, and this notebook builds it from
the parts.

This notebook grades your answers for you, and **each student gets a cell
that clears calcium at its own rate**.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON
#!pip install neuron quantities

In [ ]:
# Fetch mechanisms
# Uncomment this line if on google colab
#!git clone https://github.com/ABL-Lab/NSC6084-A26.git

In [ ]:
# Compile the mechanisms
# Note: recompiled mechanisms will not take effect until neuron is imported or the jupyter kernel is restarted

# Uncomment this line if on google colab
#!nrnivmodl ./NSC6084-A26/Sept15/mechanisms
# Uncomment this line if running locally
!nrnivmodl mechanisms

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

## Step 1b: Load your personal exercise parameters

`your_decay` is how quickly your cell clears calcium from the cytoplasm --
the single number that both questions turn on.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

soma_length = assignment.params["soma_length_um"]
g_leak      = assignment.params["g_leak_nS"]
stim_amp    = assignment.params["stim_amp_nA"]
your_decay  = assignment.params["decay_ms"]   # your calcium decay constant

print(f"Your soma length:      {soma_length} um")
print(f"Your leak conductance: {g_leak} nS")
print(f"Your current step:     {stim_amp} nA")
print(f"Exercises to submit:   {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# soma.L is YOUR personal value, loaded in Step 1b above.
# The diameter is the same for everyone: calcium influx is scaled by the
# surface-to-volume ratio 4/diam, so changing it would change the calcium
# dynamics rather than just the size of the cell.
soma.L = soma_length * um
soma.diam =  20 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

In [ ]:
area

In [ ]:
volume

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)
    sec.Ra = 100

### Add transient Na+, delayed rectified K+, leak

In [ ]:
# This model includes the transient Na+, persistent K+ and the leak conductances
soma.insert("pas")
soma.insert("NaTg")
soma.insert("K_Pst")

In [ ]:
h.celsius = 34

In [ ]:
soma(0.5).K_Pst.gK_Pstbar = 0.2
soma(0.5).NaTg.gNaTgbar = 0.42

### Parametize the leak conductance G = 1/R

In [ ]:
G = g_leak * nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
v_rest = -70*mV

In [ ]:
tau_m = (specific_membrane_capacitance * area / G).rescale(ms)

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.pas.g = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.pas.e = -70.0

In [ ]:
tau_m = ((soma(0.5).cm * uF/cm**2) / (soma(0.5).pas.g * S/cm**2)).rescale(ms)

In [ ]:
tau_m

### Inspect our parameters

In [ ]:
soma.psection()

In [ ]:
soma.nseg

### Add a current injection

In [ ]:
stim = h.IClamp(soma(0.5))

In [ ]:
stim.delay = 200 * ms    # wait 200 ms before injecting
stim.dur = 600 * ms      # a long step, so adaptation has time to develop
stim.amp = stim_amp * nA # YOUR personal step, scaled to your soma area

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

### Run the simulation

In [ ]:
h.finitialize( float(v_rest) )
h.continuerun( float(1000 * ms) )

## Step 4: Plot the results

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.xticks(size=12)
plt.yticks(size=12)
#plt.axis([0,1000,-80,50])

In [ ]:
def find_spikes(v, t):
    """ Returns times of spikes for a voltage trace and time grid"""
    # look for upward crossing of 0mV
    v_arr = np.array(v)
    t_arr = np.array(t) 
    # This is tricky & powerful notation! Let's discuss in class!
    return t_arr[1:][(v_arr[1:]>0) & (v_arr[:-1]<0)] 

### The f-I curve before any calcium

This cell fires steadily: the rate depends on the current, and once it has
settled it stays there for as long as the current is on. Keep that in mind --
it is about to stop being true.

In [ ]:
I_range = np.arange(0,0.15,0.005)

In [ ]:
def find_freq(I):
    stim.amp = I
    h.finitialize( float(v_rest) )
    h.continuerun( float(1000 * ms) )
    spike_times = find_spikes(soma_v, t)
    firing_freq = (len(spike_times)/(stim.dur*ms)).rescale(Hz)
    return firing_freq

In [ ]:
# Note this cool notation: List comprehension
freqs = [find_freq(x) for x in I_range]

In [ ]:
plt.plot(I_range, freqs, 'x')
plt.xlabel("injected current [nA]", size=14)
plt.ylabel("mean firing rate [Hz]", size=14)

In [ ]:
stim.amp = stim_amp * nA   # put your own step back after the sweep

## Adding calcium dynamics

### 1) Add extrusion and buffering

In [ ]:
soma.insert("CaDynamics_DC0")

In [ ]:
# YOUR personal calcium decay constant, loaded in Step 1b.
# This is how long it takes the cell to pump calcium back out -- Question 1
# asks you to measure it back out of the trace.
soma(0.5).CaDynamics_DC0.decay = your_decay

### Add the calcium channels

In [ ]:
soma.insert("Ca_HVA2")

In [ ]:
soma(0.5).Ca_HVA2.gCa_HVAbar = 0.005

A high-voltage-activated calcium current: it only opens during an action
potential, which is exactly what makes [Ca]i a running count of recent
spikes.

(`Ca_LVAst`, a low-voltage-activated calcium current, is also available in
the mechanisms folder if you want to explore what a *subthreshold* calcium
current does instead.)

### Insert an SK-type Ca2+ activated potassium channel

In [ ]:
soma.insert("SK_E2")

In [ ]:
soma(0.5).SK_E2.gSK_E2bar = 0.02

SK channels are opened by calcium, not by voltage. In `SK_E2.mod` the
steady-state activation is

$$ z_\infty = \frac{1}{1 + (0.00043 / [\mathrm{Ca}]_i)^{4.8}} $$

-- half-maximal at 0.43 $\mu M$, and very steep (a Hill coefficient of 4.8).
So SK is effectively a switch that watches the calcium level.

### Record the calcium and SK conductance

In [ ]:
cai = h.Vector().record(soma(0.5)._ref_cai)
gske2 = h.Vector().record(soma(0.5).SK_E2._ref_gSK_E2 )

In [ ]:
# Watch out for units
h.units("cai")

In [ ]:
h.finitialize( float(v_rest) )
h.continuerun( float(1000 * ms) )

In [ ]:
plt.plot(t, cai*1000, lw=2, label="soma(0.5).cai")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("cai [uM]", size=16)

In [ ]:
plt.plot(t, gske2, lw=2, label="soma(0.5).gSK_E2")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("specific conductance [S/cm2]", size=16)

In [ ]:
plt.plot(t, soma_v, lw=2, label="soma(0.5).v")
plt.legend(fontsize=12)
plt.xlabel("t [ms]", size=16)
plt.ylabel("v [mV]", size=16)
plt.axis([150,850,-80,50])

Three plots, one story: calcium climbs through the train, the SK conductance
follows it, and the spikes get further and further apart.

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** neuron, using the parameters printed in
Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM;
you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, fading to
zero at 20% off.

### Example Question 1 -- How fast does your cell clear calcium?
[ solved for you]

After the current step ends the spiking stops, so calcium stops coming in and
the cell simply pumps out what is left. Fit that decay and report its time
constant in **ms**.

> Subtract the resting level before taking the logarithm -- `cai` relaxes
> towards its baseline of 65 nM, not towards zero.

In [ ]:
def run_and_record():
    """ Run the standard 600 ms step and return (t, v, cai) as arrays. """
    h.finitialize( float(v_rest) )
    h.continuerun( float(1500 * ms) )
    return np.array(t), np.array(soma_v), np.array(cai)

a_t, v, ca = run_and_record()

STEP_END = 800.0           # stim.delay + stim.dur
baseline = ca[np.argmin(np.abs(a_t - 190))]   # resting [Ca]i, before the step

# Fit well after the last spike, so we are watching extrusion and nothing else.
window = (a_t >= STEP_END + 20) & (a_t <= STEP_END + 400) & ((ca - baseline) > 0)
slope, _ = np.polyfit(a_t[window], np.log(ca[window] - baseline), 1)
ca_decay_tau = -1.0 / slope

print(f"Your cell clears calcium with tau = {ca_decay_tau:.3f} ms")
assignment.submit(ca_decay_tau, "ca_decay_tau")

In [ ]:
plt.plot(a_t, (ca - baseline)*1e6, lw=2)
plt.yscale('log')
plt.axvline(STEP_END, color='k', ls='--', lw=0.8, label='step ends')
plt.xlabel("t [ms]", size=14)
plt.ylabel("[Ca]$_i$ above rest [nM]", size=14)
plt.legend()

On a log axis the decay is a straight line -- which is what makes "the"
time constant a meaningful thing to quote at all.

### Question 2 -- How much does the cell adapt?

Measure the firing rate from the **first** inter-spike interval of the step
and from the **last**, and report

$$ \frac{f_{\mathrm{last}}}{f_{\mathrm{first}}}
   = \frac{\mathrm{ISI}_{\mathrm{first}}}{\mathrm{ISI}_{\mathrm{last}}} $$

It is a ratio, so no units. A cell that does not adapt gives 1; the more it
adapts, the smaller the number.

In [ ]:
def step_isis():
    """ Inter-spike intervals during the current step, in ms. """
    a_t, v, ca = run_and_record()
    spikes = find_spikes(v, a_t)
    spikes = spikes[(spikes >= 200.0) & (spikes <= STEP_END)]
    return np.diff(spikes)

d = step_isis()
adaptation_ratio = None # fill in

print(f"{len(d)+1} spikes: {1000/d[0]:.2f} Hz at the start, "
      f"{1000/d[-1]:.2f} Hz at the end")
assignment.submit(adaptation_ratio, "adaptation_ratio")

### The control: block SK

Set `gSK_E2bar` to zero and measure the same ratio again. Without SK the
adaptation disappears entirely -- the ratio comes out at or just above 1, so
the cell fires *slightly faster* at the end of the step than at the start.

Calcium still accumulates exactly as before. It simply has nothing to act on.
That is the cleanest possible demonstration that the adaptation you measured
is the SK channel and not the calcium.

In [ ]:
soma(0.5).SK_E2.gSK_E2bar = 0.0    # block SK
d_blocked = step_isis()
print(f"SK blocked: {len(d_blocked)+1} spikes, "
      f"adaptation ratio {d_blocked[0]/d_blocked[-1]:.4f}")

soma(0.5).SK_E2.gSK_E2bar = 0.02   # put it back

### Something to think about

Compare the two voltage traces, with SK and without. The adapted cell fires
fewer spikes for the same current -- but it has also become **sensitive to
change**: a current that has been on for a while produces few spikes, while
the same current arriving fresh produces many.

A neuron with SK does not report how strong its input is. It reports how much
its input has **just changed**.